# zynarai v1 — ACE-Step 1.5 LoKr Training (Colab Pro)

**Before running:** Runtime → Change runtime type → **L4 GPU** (24 GB, Ada — don't accept a T4).
Upload your local `dataset/` folder (audio/ + dataset.json, ~2.5 GB) to Drive at
`MyDrive/lora fine tune/dataset/` before starting.

**Run order:** cells top-to-bottom. Cell **3b must run right after cell 3** — it patches a
meta-tensor bug in the model's audio tokenizer. Re-run 3b after any runtime restart/re-clone.

**Colab discipline baked in:** preprocessed tensors and checkpoints are written to Drive, so a
disconnect never costs more than the current epoch. Re-running top-to-bottom skips completed stages.

Pipeline: GPU check → mount Drive → install deps → **patch (3b)** → download base model →
copy data → preprocess (cached) → optional estimate → **LoKr training** → Gradio UI to hear results.

In [ ]:
# 1 — GPU check: want L4 (24GB) or A100. T4 = reconnect and try again.
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
import torch
cc = torch.cuda.get_device_capability()
assert cc >= (8, 0), f"compute capability {cc} < 8.0 — no Flash Attention. Get an L4/A100 runtime."
print("GPU OK")

In [ ]:
# 2 — Mount Drive + define layout
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Symlink removes the space from 'lora fine tune' so all downstream shell
# commands get a space-free path — argparse splits on spaces otherwise.
if not os.path.exists('/content/loradrive'):
    os.symlink('/content/drive/MyDrive/lora fine tune', '/content/loradrive')

DRIVE   = Path('/content/loradrive')
DATASET = DRIVE / 'dataset'          # you uploaded this (audio/ + dataset.json)
TENSORS = DRIVE / 'tensors'          # preprocessing cache (written once)
OUTPUT  = DRIVE / 'output/lokr_v1'   # training checkpoints land here (disconnect-safe)
for p in (TENSORS, OUTPUT):
    p.mkdir(parents=True, exist_ok=True)
assert (DATASET / 'dataset.json').exists(), f'upload dataset/ to Drive at: {DATASET}'
n_audio = len(list((DATASET / 'audio').glob('*')))
print(f'dataset found: {n_audio} audio files')
print(f'DRIVE  = {DRIVE}  (exists={DRIVE.exists()})')
print(f'TENSORS= {TENSORS}')
print(f'OUTPUT = {OUTPUT}')

In [ ]:
# 3 — Install ACE-Step 1.5 + Side-Step deps (~3 min, no compile)
# We FILTER requirements.txt to avoid two hangs:
#   (a) flash-attn — its linux entry compiles from source (20-40 min). SDPA fallback is fine.
#   (b) torch==2.10.0+cu128 — would downgrade Colab's working torch 2.11 (~1 GB re-download).
%cd /content
![ -d ACE-Step-1.5 ] || git clone --depth 1 https://github.com/ace-step/ACE-Step-1.5.git
%cd /content/ACE-Step-1.5

# strip torch pins, flash-attn, the pytorch extra indexes, and triton (Colab has it)
lines = open('requirements.txt').read().splitlines()
SKIP = ('torch==', 'torchvision==', 'torchaudio==', 'flash-attn', '--extra-index-url', 'triton')
keep = [l for l in lines if not any(tok in l for tok in SKIP)]
open('/tmp/req_colab.txt', 'w').write('\n'.join(keep))
n = sum(1 for l in keep if l.strip() and not l.strip().startswith('#'))
print(f'installing {n} requirement lines (torch + flash-attn excluded)')

!pip install -q -r /tmp/req_colab.txt
!pip install -q -r requirements-sidestep.txt
!pip install -q loguru lycoris-lora vector_quantize_pytorch
print("deps installed — SDPA backend active, Colab torch kept")

In [ ]:
# 3b — Runtime patches (RE-RUN after any runtime restart or re-clone)
# Newer transformers instantiates trust_remote_code models on the 'meta'
# device, but vector_quantize_pytorch asserts on tensor values in __init__,
# which is illegal on meta tensors. Guard those asserts + force CPU load.
import os, re, vector_quantize_pytorch

vq_dir = os.path.dirname(vector_quantize_pytorch.__file__)
for fname in os.listdir(vq_dir):
    if not fname.endswith('.py'):
        continue
    fp = os.path.join(vq_dir, fname)
    t = open(fp).read()
    # guard `assert (<expr> > 1).all()` against meta-device tensors
    t2 = re.sub(
        r'assert \(([^)]+?) > 1\)\.all\(\)',
        r"assert \1.device.type == 'meta' or (\1 > 1).all()",
        t,
    )
    if t2 != t:
        open(fp, 'w').write(t2)
        print(f'patched {fname}')

# model_loader: force real (non-meta) weights via low_cpu_mem_usage=False
ml = '/content/ACE-Step-1.5/acestep/training_v2/model_loader.py'
t = open(ml).read()
if 'low_cpu_mem_usage=False' not in t:
    t = t.replace(
        '                attn_implementation=attn_impl,\n                dtype=dtype,\n            )',
        '                attn_implementation=attn_impl,\n                dtype=dtype,\n                low_cpu_mem_usage=False,\n            )',
    )
    open(ml, 'w').write(t)
    print('model_loader patched')
else:
    print('model_loader already patched')
print('done — safe to run preprocessing / training')


In [ ]:
# 4 — Download checkpoints (~25GB to fast local disk; re-downloads per session)
# The main repo ships VAE + text encoder + LM + the TURBO DiT only; the
# base-variant DiT is a separate repo, downloaded into a sibling dir so
# model_discovery detects it as variant 'base'.
CKPT = '/content/checkpoints'
from huggingface_hub import snapshot_download
snapshot_download('ACE-Step/Ace-Step1.5', local_dir=CKPT)
snapshot_download('ACE-Step/acestep-v15-base', local_dir=f'{CKPT}/acestep-v15-base')
!ls {CKPT}

In [ ]:
# 5 — Copy dataset from Drive to local disk (Drive I/O is slow for audio decode)
import shutil, os
src = str(DATASET)
dst = '/content/data'
os.makedirs(dst, exist_ok=True)
shutil.copytree(src, dst, dirs_exist_ok=True)
audio_files = list(Path(dst + '/audio').glob('*'))
print(f"copied {len(audio_files)} audio files to {dst}")

In [ ]:
# 6 — Preprocess audio -> training tensors (skipped only if FINISHED .pt files exist)
# Two-pass: Pass 1 writes <name>.tmp.pt, Pass 2 finalizes <name>.pt and deletes the tmp.
# The guard counts real .pt files (excluding .tmp.pt) so a half-done run re-runs cleanly.
# cwd='/content' so the trainer's safe_path guard accepts Drive paths.
import subprocess
from pathlib import Path
tensors_dir = str(TENSORS)
final_pt = [f for f in Path(tensors_dir).glob('*.pt') if not f.name.endswith('.tmp.pt')]
if len(final_pt) > 0:
    print(f'{len(final_pt)} finished tensor files found — skipping preprocessing')
else:
    # clear any leftover half-done intermediates before a fresh run
    for f in Path(tensors_dir).glob('*.tmp.pt'):
        f.unlink()
    result = subprocess.run([
        'python', '/content/ACE-Step-1.5/train.py', 'fixed',
        '--checkpoint-dir', CKPT,
        '--model-variant', 'base',
        '--dataset-dir', tensors_dir,
        '--output-dir', str(OUTPUT),
        '--preprocess',
        '--audio-dir', '/content/data/audio',
        '--dataset-json', '/content/data/dataset.json',
        '--tensor-output', tensors_dir,
    ], cwd='/content', capture_output=True, text=True)
    print(result.stdout[-4000:] if result.stdout else '')
    if result.returncode != 0:
        print('--- STDERR ---')
        print(result.stderr[-4000:])
        raise RuntimeError(f'train.py exited {result.returncode}')
final_pt = [f for f in Path(tensors_dir).glob('*.pt') if not f.name.endswith('.tmp.pt')]
print(f'{len(final_pt)} finished .pt tensor files in cache')

In [ ]:
# 7 (optional) — Gradient sensitivity estimate (~2 min): which modules matter most
import subprocess
r = subprocess.run([
    'python', '/content/ACE-Step-1.5/train.py', 'estimate',
    '--checkpoint-dir', CKPT,
    '--model-variant', 'base',
    '--dataset-dir', str(TENSORS),
    '--estimate-batches', '5', '--top-k', '16',
], cwd='/content', capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print('--- STDERR ---'); print(r.stderr[-3000:])

In [ ]:
# 8 — LoKr training. 57 tracks -> Side-Step guidance is 50-100 epochs for 50+ songs.
#     --yes skips the interactive prompt; cwd='/content' satisfies the safe_path guard.
#     SMOKE TEST FIRST: set epochs=5, confirm loss drops + a checkpoint lands, then raise to 100.
#     Checkpoints save to Drive every N epochs; resume with --resume-from if disconnected.
import subprocess
subprocess.run([
    'python', '/content/ACE-Step-1.5/train.py', '--yes', 'fixed',
    '--checkpoint-dir', CKPT,
    '--model-variant', 'base',
    '--dataset-dir', str(TENSORS),
    '--output-dir', str(OUTPUT),
    '--adapter-type', 'lokr',
    '--lokr-linear-dim', '64',
    '--lokr-linear-alpha', '128',
    '--lokr-factor', '-1',
    '--lokr-weight-decompose',
    '--learning-rate', '0.03',
    '--epochs', '100',
    '--batch-size', '2',
    '--gradient-accumulation', '2',
    '--save-every', '10',
    '--seed', '42',
], cwd='/content', check=True)

**If the session disconnects mid-run:** re-run cells 1–3b (deps + patch), then 4–6
(preprocessing skips via the Drive cache), then re-run the training cell with
`--resume-from` pointing at the latest checkpoint in
`MyDrive/lora fine tune/output/lokr_v1/`.

**Smoke test first!** For the very first run, change `--epochs 100` to `--epochs 5` and
confirm the whole pipeline completes and loss decreases before committing to the full run.

In [ ]:
# 9 — Hear it: launch the ACE-Step Gradio UI with a public link.
#     NOTE: the launcher is `acestep.acestep_v15_pipeline` (not api_server, which is headless).
#     In the UI: Service Configuration -> load model (base) -> load your LoKr adapter from
#     /content/loradrive/output/lokr_v1/ (epoch-100) -> fixed seed -> A/B with vs without trigger:
#       "zynarai, driving tech house, gritty bassline pressure, dark moody tone"
#       "driving tech house, gritty bassline pressure, dark moody tone"   (control)
import os
os.environ['ACESTEP_CHECKPOINT_DIR'] = CKPT
!python -m acestep.acestep_v15_pipeline --share

## Evaluation checklist (from PLAN.md phase 4)

1. Fixed 10-prompt A/B: trigger word vs no trigger word, same seeds.
2. Overfit check: does it regurgitate recognizable melodies from the 57 (bad), or capture
   texture/groove/tonality (good)? Compare epoch-50 vs epoch-100 checkpoints — earlier
   often generalizes better.
3. Dilution reality-check: given the v1 captions, expect `underground warehouse energy` and
   `clean modern club mix` to do little (they're near-constants); the axes that stayed
   contrastive (`weighty kick-driven low end`, `busy layered percussion`, R&B hooks) are the
   dials most worth testing.
4. Download the final adapter from Drive; it also runs locally in the macOS Gradio UI.
5. Write up labeling-effort + quality notes -> feeds the v2 (Zynar) plan and the startup thesis.